In [1]:
"""
Verifica se todas as sequências em cada dataset.jsonl têm o mesmo comprimento.
Roda tanto para benchmark_prepared_train quanto para benchmark_prepared_test.
"""

import json
import os
from collections import Counter
from pathlib import Path

DATASETS = {
    "cif_2016_filtered":        {"subfolder": "horizon_12", "horizon": 12},
    "etth_filtered":            {"subfolder": "horizon_36", "horizon": 36},
    "hospital_filtered":        {"subfolder": "horizon_12", "horizon": 12},
    "m3_monthly_filtered":      {"subfolder": "horizon_18", "horizon": 18},
    "m4_monthly_filtered":      {"subfolder": "horizon_18", "horizon": 18},
    "nn5_weekly_filtered":      {"subfolder": "horizon_8",  "horizon": 8},
    "tourism_monthly_filtered": {"subfolder": "horizon_24", "horizon": 24},
    "weather_filtered":         {"subfolder": "horizon_36", "horizon": 36},
    "fred_md_filtered":         {"subfolder": "horizon_12",  "horizon": 12},
    "m5_filtered":              {"subfolder": "horizon_28",  "horizon": 28},
}

BASE_PATHS = ["benchmark_prepared_train", "benchmark_prepared_test"]


def check_jsonl(path: Path):
    """
    Lê um arquivo .jsonl e retorna:
      - total de linhas
      - Counter com {comprimento: quantidade de linhas}
      - lista com (numero_da_linha, comprimento) das que diferem do mais comum
    """
    length_counter = Counter()
    line_lengths = []  # (line_no, length)

    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"    [ERRO] Linha {i} não é JSON válido: {e}")
                continue

            seq = obj.get("sequence")
            if seq is None:
                print(f"    [ERRO] Linha {i} não possui campo 'sequence'.")
                continue

            length = len(seq)
            length_counter[length] += 1
            line_lengths.append((i, length))

    return length_counter, line_lengths


def report(base_path: str):
    print("=" * 80)
    print(f"VERIFICANDO: {base_path}")
    print("=" * 80)

    base = Path(base_path)
    if not base.exists():
        print(f"[AVISO] Caminho não encontrado: {base.resolve()}")
        return

    overall_ok = True

    for ds_name, info in DATASETS.items():
        sub = info["subfolder"]
        horizon = info["horizon"]
        jsonl_path = base / ds_name / sub / "dataset.jsonl"

        print(f"\n>> Dataset: {ds_name}  (subfolder={sub}, horizon={horizon})")
        print(f"   Arquivo: {jsonl_path}")

        if not jsonl_path.exists():
            print("   [AVISO] Arquivo não encontrado, pulando.")
            overall_ok = False
            continue

        counter, line_lengths = check_jsonl(jsonl_path)
        total = sum(counter.values())

        if total == 0:
            print("   [AVISO] Nenhuma sequência válida encontrada.")
            overall_ok = False
            continue

        unique_lengths = sorted(counter.keys())
        print(f"   Total de linhas: {total}")
        print(f"   Comprimentos únicos encontrados: {unique_lengths}")

        if len(unique_lengths) == 1:
            L = unique_lengths[0]
            print(f"   ✅ Todas as {total} linhas têm comprimento = {L}")
        else:
            overall_ok = False
            print(f"   ❌ Comprimentos diferentes detectados!")
            print("   Distribuição (comprimento -> quantidade):")
            for L, qty in sorted(counter.items()):
                print(f"      {L}: {qty}")

            # Mostra alguns exemplos divergentes (até 5)
            most_common_len = counter.most_common(1)[0][0]
            divergentes = [
                (i, L) for (i, L) in line_lengths if L != most_common_len
            ]
            print(f"   Comprimento mais comum: {most_common_len}")
            print(f"   Linhas divergentes (até 5 primeiras):")
            for i, L in divergentes[:5]:
                print(f"      linha {i}: comprimento {L}")

    print()
    print("-" * 80)
    if overall_ok:
        print(f"RESUMO {base_path}: ✅ todos os datasets têm sequências de comprimento uniforme.")
    else:
        print(f"RESUMO {base_path}: ⚠️  há datasets com problemas — veja acima.")
    print("-" * 80)


if __name__ == "__main__":
    for bp in BASE_PATHS:
        report(bp)
        print()

VERIFICANDO: benchmark_prepared_train

>> Dataset: cif_2016_filtered  (subfolder=horizon_12, horizon=12)
   Arquivo: benchmark_prepared_train\cif_2016_filtered\horizon_12\dataset.jsonl
   Total de linhas: 48
   Comprimentos únicos encontrados: [108]
   ✅ Todas as 48 linhas têm comprimento = 108

>> Dataset: etth_filtered  (subfolder=horizon_36, horizon=36)
   Arquivo: benchmark_prepared_train\etth_filtered\horizon_36\dataset.jsonl
   Total de linhas: 200
   Comprimentos únicos encontrados: [138]
   ✅ Todas as 200 linhas têm comprimento = 138

>> Dataset: hospital_filtered  (subfolder=horizon_12, horizon=12)
   Arquivo: benchmark_prepared_train\hospital_filtered\horizon_12\dataset.jsonl
   Total de linhas: 767
   Comprimentos únicos encontrados: [72]
   ✅ Todas as 767 linhas têm comprimento = 72

>> Dataset: m3_monthly_filtered  (subfolder=horizon_18, horizon=18)
   Arquivo: benchmark_prepared_train\m3_monthly_filtered\horizon_18\dataset.jsonl
   Total de linhas: 348
   Comprimentos úni